# Alfa Train — Kaggle GPU (T4x2)
Colab limit শেষ? এখানে চালান। **আগে 3টা setting:**
1. ডান প্যানেল **Session options → Accelerator → GPU T4x2**
2. **Internet → ON** (pip + HuggingFace-এর জন্য must — OFF থাকলে name-resolution error আসবে)
3. **Add-ons → Secrets → Add secret**: Name=`HF_TOKEN`, Value=HF Write token, notebook-এ attach ✓
Free quota: ~30h GPU/week, session ~12h। Model বাঁচাতে শেষে **Save Version** বা Hub upload করুন।


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers datasets tokenizers accelerate huggingface_hub pyyaml bitsandbytes


In [ ]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Right panel Session options -> Accelerator -> GPU T4x2, then restart session.')


In [ ]:
tok = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('secret note:', e)
if not tok:
    import os
    tok = os.getenv('HF_TOKEN')
if tok:
    from huggingface_hub import login
    login(token=tok)
    print('logged in')
else:
    print('No HF_TOKEN secret — Hub upload step will be skipped. Add-ons -> Secrets -> HF_TOKEN.')


In [ ]:
# Code from public GitHub (no token needed)
%cd /kaggle/working
!rm -rf /kaggle/working/alfa
!git clone https://github.com/mrsandipmandal/alfa-code.git /kaggle/working/alfa
%cd /kaggle/working/alfa
!ls scripts/ configs/


In [ ]:
# Pick model ONCE here — every cell below follows it.
MODEL = "base-1B"  # or "tiny-50M" for the fast test
# FSDP: shard model across BOTH T4s (~2.5-3x faster). Needs 2 GPUs (Kaggle T4x2).
# False = single GPU (safe everywhere, incl. Colab T4).
USE_FSDP = True
if MODEL == "base-1B":
    CONFIG, OUT, EPOCHS, SEQ_LEN, BS, ROWS, CHAT_ROWS = (
        "configs/base-1B.yaml", "/kaggle/working/outputs/base-1B", 3, 1024, 1, 5000, 2000)
else:
    CONFIG, OUT, EPOCHS, SEQ_LEN, BS, ROWS, CHAT_ROWS = (
        "configs/tiny-50M.yaml", "/kaggle/working/outputs/tiny-50M", 1, 2048, 2, 1500, 500)
if USE_FSDP:
    LAUNCH, SEQ_LEN, BS, ACCUM, FSDP_FLAG = ("accelerate launch --num_processes 2", 2048, 2, 8, "--fsdp full_shard")
else:
    LAUNCH, ACCUM, FSDP_FLAG = ("CUDA_VISIBLE_DEVICES=0 python", 16, "")
print("MODEL:", MODEL, "| OUT:", OUT, "| epochs:", EPOCHS, "| seq:", SEQ_LEN, "| bs:", BS, "| accum:", ACCUM, "| rows:", ROWS, "| chat:", CHAT_ROWS)
print("LAUNCH:", LAUNCH, "| FSDP:", FSDP_FLAG or "off")


In [ ]:
!python scripts/download_data.py --dataset multimodal_mix --max-rows {ROWS} --overwrite
!python scripts/download_data.py --dataset chat_qa --max-rows {CHAT_ROWS}
!python scripts/train_tokenizer.py
!ls -lh tokenizers/ data/processed/


In [ ]:
!{LAUNCH} scripts/train_hf.py --data data/processed/train.jsonl --out {OUT} --epochs {EPOCHS} --config {CONFIG} --max-seq-len {SEQ_LEN} --batch-size {BS} --grad-accum {ACCUM} {FSDP_FLAG}
!ls -lh {OUT}


In [ ]:
# SELF-LEARNING CYCLE: generate -> filter -> append -> versioned retrain.
CYCLE = 1  # bump every round: 1, 2, 3...
!python scripts/self_learn.py --cycle {CYCLE} --ckpt {OUT} --topics data/seeds/topics.txt --from-train 100 --n 200 --max-tokens 256 --append
!{LAUNCH} scripts/train_hf.py --data data/processed/train.jsonl --out /kaggle/working/outputs/self-cycle-{CYCLE} --epochs 1 --resume-from {OUT} --config {CONFIG} --max-seq-len {SEQ_LEN} --batch-size {BS} --grad-accum {ACCUM} {FSDP_FLAG}
!ls -lh /kaggle/working/outputs/self-cycle-{CYCLE}
# review, then promote:  !cp -r /kaggle/working/outputs/self-cycle-{CYCLE} {OUT}


In [ ]:
# Push to Hub (uploads every trained model found).
import os
from pathlib import Path
tok = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    tok = os.getenv('HF_TOKEN')
if not tok:
    print('SKIP: no HF_TOKEN secret set.')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=tok)
    found = sorted(p for p in Path('/kaggle/working/outputs').glob('*') if (p / 'config.json').exists())
    if not found:
        print('nothing trained yet in /kaggle/working/outputs')
    for local in found:
        api.upload_folder(folder_path=str(local), repo_id='mrsandip/Alfa-Code', path_in_repo=f'outputs/{local.name}')
        print('uploaded', local.name)


In [ ]:
# Inference sample test (uses the trained OUT above).
!python scripts/infer.py --local {OUT} --prompt "def fib(n):" --wrap --max-new-tokens 200
# Persistence: File -> Save Version (keeps /kaggle/working), and/or Hub upload cell above.
